In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [28]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [5]:
loader = PyPDFLoader("../data/medical_report (1).pdf")
docs = loader.load()

In [6]:
len(docs)

9

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)

In [8]:
len(splitted_docs)

26

In [9]:
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
vector_store= InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [12]:
same_record = vector_store.similarity_search("Patient name")

In [17]:
# agent = tools,llm, prompt

In [33]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve data of the PDF Documents, and these PDF documents have details about medical reports.
    """

    print("Tool Called", query)
    docs = vector_store.similarity_search(query=query, k=4)
    context = ""
    for doc in docs:
        context = doc.page_content + "\n\n"
    return context
   

In [24]:
retriever_tool.invoke("Patient Name")

'Consultant- Hemato-Oncopathology & \nFlowcytometry                                     \nNRL - Dr Lal PathLabs Ltd\nDr Sunanda\nMD, Pathology\nSr. Consultant Pathologist - \nHematology & Immunology                              \nNRL - Dr Lal PathLabs Ltd\n-------------------------------End of report --------------------------------\nAHEEEHAPMKHJBKHNLKIHBLLCBILLJCECCJLCIKPLPKEDFBFAPPAHEEEHA\nBNFFFNBPAPBOACIGFGELNGAPAOAHFHAKAKNOBCJKBLEKMCJGNPBNFFFNB\nCIEGCAFJLPNBKNNOPFIAAJHFJGDEHEDFLKPHENFLMLKIOEFLBKHDEHANP\nDJECMEFNBMHIEDINMIMELFMFAFPLBLAFJIFFOAPAKCLJPDNLIJFBKEMEK\nKDCILHFJONFFDIAJMBICKPJDCENJKPCHKKMBACOIPLKGKPNGKFFFOIOND\nFGCBBEFNNICGDKHKILIGCIPBADHFOFAINCFCBKDKBLIKOMNALNEJOGEKD\nDPIMIJFNELODDJHIEKMPCGFHIHAFJBALPPMPBLMIKJCHKJNKIJNFMDILD\nNJJJAMFMAKNKPELJGANHDOMILLBMFMBHIFCCAMNNIIKNOPNKBNFMBHILL\nNKIHJAFMDKOGFAAMFNOIHMIKHEJFEKFBJFFKBKOFOOBIOCNKDJPHNEKLJ\nNMDBPJFCEHNJJDCLCDBFGPNFIDFKPNJHKEPLELPLNKCKIGEOCIGKHMEJP\nICIKOLFLGKCBMMCMPNBDAJEEINMEFCBBJMFEBFOEONDMKPFKEPCIGKAMG\n\n'

In [25]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [26]:
System_Prompt = """
    You are a helpful assistant that answers questions using retreived context.
    ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [35]:
agent = create_agent(
    model= llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [36]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called patient name doctor name medical report


In [37]:
result = response["messages"][-1].content

In [38]:
print(result)

**Patient Name:** Ms. Nikita Chudhary  

**Doctors Mentioned:**  
- Dr. Nitin Nahar  
- Dr. Swapnil Gupta, MD (Pathology) – Chief of Laboratory, Dr Lal PathLabs Ltd.  
- Dr. Kiran Bhargava Pathak, MD (Pathology) – Chief of Laboratory, Dr Lal PathLabs Ltd.  

These names are taken from the report header and the “Test Report” section of the PDF.
